In [63]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(project_root)

c:\Users\Макетолог\Документы\MYPROJECT\data-lab


In [55]:
from datetime import date, timedelta

from src.common.ch_client import ch_select, ch_execute
from src.pipelines.cbr_pipeline import get_cbr_rates_history
from src.load.clickhouse_load import (
    delete_currency_rates_by_source_date,
    load_currency_rates_to_clickhouse
)

In [40]:
import importlib
import src.load.clickhouse_load

importlib.reload(src.load.clickhouse_load)

from src.load.clickhouse_load import load_currency_rates_to_clickhouse

In [58]:
end_date = date.today()
start_date = end_date - timedelta(days=90)


In [42]:
currency_rates_history_df = get_cbr_rates_history(
    start_date,
    end_date
)

In [6]:
ch_select(
    """
    describe table data_lab.raw_cbr_rates
    """
)

,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,load_dttm,DateTime,,,,,
1,sourse_date,Date,,,,,
2,source_date,Date,,,,,
3,rate_date,Date,,,,,
4,currency_code,String,,,,,
5,currency_name,String,,,,,
6,nominal,UInt32,,,,,
7,rate,Float64,,,,,


In [7]:
ch_execute(
    """
    alter table data_lab.raw_cbr_rates
    add column if not exists sourse_date Date
    after load_dttm
"""
)

In [64]:
delete_result = delete_currency_rates_by_source_date(
    start_date,
    end_date
)

delete_result

{'table_name': 'data_lab.raw_cbr_rates',
 'start_date': datetime.date(2026, 6, 16),
 'end_date': datetime.date(2026, 7, 16),
 'status': 'success'}

In [47]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 2052,
 'status': 'success'}

In [54]:
row_count_df = ch_select("""
    SELECT
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
""")

row_count_df

,row_count
0,7398


In [46]:
duplicate_check_df = ch_select("""
    SELECT
        source_date,
        currency_code,
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
    GROUP BY
        source_date,
        currency_code
    HAVING row_count > 1
    ORDER BY
        source_date,
        currency_code
""")

duplicate_check_df

""


In [18]:
from src.quality.cbr_checks import validate_cbr_rates_df

In [53]:
validation_result = validate_cbr_rates_df(
    currency_rates_history_df
)

validation_result

{'status': 'success', 'row_count': 2052, 'errors': []}

In [52]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 2052,
 'status': 'success'}

In [51]:
broken_currency_rates_df = currency_rates_history_df.copy()

broken_currency_rates_df.loc[
    0,
    'rate'
] = -1

In [22]:
load_currency_rates_to_clickhouse(
    broken_currency_rates_df
)

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 0,
 'status': 'failed',
 'validation_errors': ['Invalid rate values: 1']}

In [65]:
from datetime import date, timedelta
from src.pipelines.cbr_pipeline import run_cbr_rates_pipeline

end_date=date.today()
start_date=end_date - timedelta(days=30)

pipeline_result = run_cbr_rates_pipeline(
    start_date=start_date,
    end_date=end_date
)

pipeline_result

{'pipeline_name': 'cbr_rates_pipeline',
 'start_date': datetime.date(2026, 6, 16),
 'end_date': datetime.date(2026, 7, 16),
 'rows_loaded': 1674,
 'status': 'success',
 'delete_result': {'table_name': 'data_lab.raw_cbr_rates',
  'start_date': datetime.date(2026, 6, 16),
  'end_date': datetime.date(2026, 7, 16),
  'status': 'success'},
 'load_result': {'table_name': 'data_lab.raw_cbr_rates',
  'rows_loaded': 1674,
  'status': 'success'}}